## Multipurpose Text Webscraper

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from collections import deque
import time
import re

def validate_url(url):
    """Validate that the URL is properly formatted."""
    url_pattern = re.compile(
        r'^https?://'  # http:// or https://
        r'(?:(?:[A-Z0-9](?:[A-Z0-9-]{0,61}[A-Z0-9])?\.)+[A-Z]{2,6}\.?|'  # domain
        r'localhost|'  # localhost
        r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})'  # IP address
        r'(?::\d+)?'  # optional port
        r'(?:/?|[/?]\S+)$', re.IGNORECASE)
    return url_pattern.match(url) is not None

def get_user_input():
    """Get website URL and options from user."""
    
    print("\n" + "="*80)
    print("WEBSITE TEXT SCRAPER")
    print("="*80)
    
    # Get URL
    while True:
        url = input("\nEnter the website URL (e.g., https://example.com): ").strip()
        
        if not url:
            print("URL cannot be empty.")
            continue
        
        # Add https:// if missing
        if not url.startswith(('http://', 'https://')):
            url = 'https://' + url
        
        if validate_url(url):
            break
        else:
            print("Invalid URL format. Please try again.")
    
    # Get domain confirmation
    domain = urlparse(url).netloc
    print(f"\nDomain detected: {domain}")
    confirm = input("Scrape only this domain? (y/n, default=y): ").strip().lower()
    
    if confirm == 'n':
        custom_domain = input("Enter custom domain to limit to: ").strip()
        if custom_domain:
            domain = custom_domain
    
    # Get delay preference
    print("\nRecommended: 1-2 second delay between requests (be respectful to servers)")
    delay = input("Enter delay in seconds (default=1): ").strip()
    try:
        delay = float(delay) if delay else 1
    except ValueError:
        delay = 1
    
    # Get output preference
    save_to_file = input("\nSave output to file? (y/n, default=y): ").strip().lower()
    save_to_file = save_to_file != 'n'
    
    # Get page limit
    print("\nLimit pages to scrape? (prevents extremely large crawls)")
    limit = input("Max pages (0 for unlimited, default=0): ").strip()
    try:
        limit = int(limit) if limit else 0
    except ValueError:
        limit = 0
    
    return url, domain, delay, save_to_file, limit

def scrape_website(start_url, domain=None, delay=1, max_pages=0):
    """
    Scrape all text from a website.
    
    Args:
        start_url: The starting URL
        domain: Domain to limit crawling to
        delay: Delay between requests in seconds
        max_pages: Maximum pages to scrape (0 = unlimited)
    """
    
    if domain is None:
        domain = urlparse(start_url).netloc
    
    visited = set()
    to_visit = deque([start_url])
    all_text = {}
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    print(f"\nStarting scrape of {start_url}")
    print(f"Domain: {domain}")
    print(f"Delay: {delay}s between requests")
    print("-" * 80)
    
    while to_visit:
        # Check page limit
        if max_pages > 0 and len(visited) >= max_pages:
            print(f"\nReached max page limit ({max_pages})")
            break
        
        url = to_visit.popleft()
        
        if url in visited:
            continue
        
        if urlparse(url).netloc != domain:
            continue
        
        visited.add(url)
        
        try:
            print(f"[{len(visited)}] Scraping: {url}")
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Extract text
            text = soup.get_text(separator='\n', strip=True)
            all_text[url] = text
            
            # Find all links
            for link in soup.find_all('a', href=True):
                href = link['href']
                absolute_url = urljoin(url, href).split('#')[0]
                
                if urlparse(absolute_url).netloc == domain and absolute_url not in visited:
                    to_visit.append(absolute_url)
            
            time.sleep(delay)
            
        except requests.RequestException as e:
            print(f"Error scraping {url}: {e}")
    
    return all_text

def save_results(content, filename="scraped_content.txt"):
    """Save scraped content to file."""
    try:
        with open(filename, "w", encoding="utf-8") as f:
            for url, text in content.items():
                f.write(f"\n{'='*80}\n")
                f.write(f"URL: {url}\n")
                f.write(f"{'='*80}\n")
                f.write(text)
                f.write("\n\n")
        print(f"\nContent saved to {filename}")
    except Exception as e:
        print(f"Error saving file: {e}")

# Main execution
if __name__ == "__main__":
    try:
        url, domain, delay, save_file, max_pages = get_user_input()
        
        content = scrape_website(url, domain, delay, max_pages)
        
        print(f"\n" + "="*80)
        print(f"Scraped {len(content)} pages successfully")
        print("="*80)
        
        # Show preview
        if content:
            first_url = list(content.keys())[0]
            first_text = content[first_url][:300]
            print(f"\nPreview of first page:\n{first_text}...\n")
        
        # Save if requested
        if save_file and content:
            save_results(content)
        
        # Print URLs scraped
        print("\nPages scraped:")
        for url in content.keys():
            print(f"  - {url}")
    
    except KeyboardInterrupt:
        print("\n\nScraping cancelled by user.")
    except Exception as e:
        print(f"\nUnexpected error: {e}")


WEBSITE TEXT SCRAPER

Domain detected: chargeviz.com

Recommended: 1-2 second delay between requests (be respectful to servers)

Limit pages to scrape? (prevents extremely large crawls)

Starting scrape of https://chargeviz.com/
Domain: chargeviz.com
Delay: 1.0s between requests
--------------------------------------------------------------------------------
[1] Scraping: https://chargeviz.com/
[2] Scraping: https://chargeviz.com/deploy/
[3] Scraping: https://chargeviz.com/monitor/
[4] Scraping: https://chargeviz.com/about/
[5] Scraping: https://chargeviz.com/tesla-supercharger-pricing-in-europe/
[6] Scraping: https://chargeviz.com/mentions-legales/

Scraped 6 pages successfully

Preview of first page:
ChargeViz｜First Yield Management Platform for CPOs
Home
Products
Deploy
Understand patterns.
Monitor
Boost traffic
Optimize
in coming
Local/dynamic pricing + energy
Company
About
Why ChargeViz ?
Insights
Tesla Supercharger Pricing (Europe)
Contact
Email us
Optimize EV Charging revenue
T